# IO handler from time series

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import glob
import os
import shutil

import affine
import numpy as np
import pandas as pd
import rioxarray as rio  # noqa: F401
import xarray as xr
from pyproj import CRS

In [ ]:
import sweat.timeseries.io_handler as ioh
import sweat.timeseries.status_handler as sh
from sweat.common.io import read_data_from_file, write_dataset
from sweat.timeseries.types import TimeSeriesVar as TSVar

### Create time series

In [ ]:
# Setup data
window_size = 7  # Time
x_size = 2
y_size = 1
today = dt.datetime.now(tz=dt.UTC).date()
dates = np.array(pd.date_range(end=today, periods=window_size).to_list())
# Define transform: (origin_x, origin_y), pixel size = 0.1 degree
transform = affine.Affine.translation(0, 40) * affine.Affine.scale(0.1, -0.1)
# Generate x/y coordinates from transform
x_coords = [(transform * (i, 0))[0] for i in range(x_size)]  # x: lon
y_coords = [(transform * (0, j))[1] for j in range(y_size)]  # y: lat
# Create a dataset
et_ts = xr.Dataset(
    {
        TSVar.ET.value: (
            ["y", "x", "time"],
            np.array(
                [
                    [
                        [1.2, 1.4, 1.6, 1.8, 1.8, 1.8],
                        [1.4, 1.4, 1.4, 1.4, 1.4, 1.4],
                    ]
                ]
            ),
        ),
        TSVar.FLAGS.value: (
            ["y", "x", "time"],
            np.array(
                [
                    [
                        [16, 816, 816, 16, 336, 592],
                        [368, 0, 336, 592, 880, 1136],
                    ]
                ],
                dtype=sh.STATUS_TYPE,
            ),
        ),
    },
    coords={
        "time": dates[:-1],
        "y": y_coords,
        "x": x_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
et_ts = et_ts.rio.write_crs(CRS(4236))
et_ts = et_ts.rio.write_transform(transform)
# Create a radiation dataset
radiation_ts = xr.Dataset(
    {
        TSVar.RADIATION.value: (
            ["y", "x", "time"],
            np.array(
                [
                    [
                        [200, 200, 200, 200, 200, 200, 200],
                        [200, 200, 200, 200, 200, 200, 200],
                    ]
                ]
            ),
        ),
    },
    coords={
        "time": dates,
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
radiation_ts = radiation_ts.rio.write_crs(CRS(4236))
radiation_ts = radiation_ts.rio.write_transform(transform)
# Create a dataset for ET single dates
acquisition_dates = [1, 3, 5, 6]
feed_dates = dates[acquisition_dates]
feed = xr.Dataset(
    {
        TSVar.ET.value: (
            ["y", "x", "time"],
            np.array(
                [[[np.nan, 1.8, np.nan, 2.4], [1.4, np.nan, 2.2, np.nan]]]
            ),
        ),
        TSVar.FLAGS.value: (
            ["y", "x", "time"],
            np.array([[[0, 1, 0, 1], [1, 0, 1, 0]]]),
        ),
    },
    coords={
        "time": feed_dates,
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
feed = feed.rio.write_crs(CRS(4236))
feed = feed.rio.write_transform(transform)
# Create a DEM dataset
dem = xr.Dataset(
    {
        TSVar.HEIGHT.value: (
            ["y", "x"],
            np.array([[10.0, 10.0]]),
        ),
        TSVar.SLOPE.value: (
            ["y", "x"],
            np.array([[5.0, 5.0]]),
        ),
        TSVar.ASPECT.value: (
            ["y", "x"],
            np.array([[10.0, 10.0]]),
        ),
    },
    coords={
        "y": y_coords,
        "x": x_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
dem = dem.rio.write_crs(CRS(4236))
dem = dem.rio.write_transform(transform)

### Write time series 

In [ ]:
output_dir = os.path.join(os.getcwd(), "out")
if os.path.isdir(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
ioh.write_timeseries(
    et_ts,
    root_name="et_time_series",
    directory=output_dir,
)
ioh.write_timeseries(radiation_ts, root_name="radiation", directory=output_dir)
ioh.write_timeseries(feed, root_name="et_single_date", directory=output_dir)
write_dataset(dem, filename="dem.tif", directory=output_dir)

### Read time series

In [ ]:
# Generate lists of file paths
et_ts_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("et_time_series_*.tif", root_dir=output_dir)
]
radiation_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("radiation_*.tif", root_dir=output_dir)
]
et_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("et_single_date_*.tif", root_dir=output_dir)
]
dem_file = os.path.join(output_dir, "dem.tif")

In [ ]:
print("ET time series = ", et_ts_files)
print("Radiation = ", radiation_files)
print("ET single date = ", et_files)
print("DEM = ", dem_file)

In [ ]:
dates

In [ ]:
et_ts = ioh.read_et_time_series(et_ts_files, dates=dates)

In [ ]:
et_ts

In [ ]:
et_ts.et.dims

In [ ]:
radiation_ts = ioh.read_radiation_time_series(radiation_files)

In [ ]:
radiation_ts

In [ ]:
radiation_ts.daily_radiation.dims

In [ ]:
et_feed = ioh.read_et_single_date(et_files)

In [ ]:
et_feed

In [ ]:
dem = read_data_from_file(dem_file)

In [ ]:
dem

### Configure timeseries and read input data

In [ ]:
config = {
    "dates": [
        "2025-08-23",
        "2025-08-24",
        "2025-08-25",
        "2025-08-26",
        "2025-08-27",
        "2025-08-28",
        "2025-08-29",
    ],
    "et_time_series": et_ts_files,
    "radiation": radiation_files,
    "et_single_date": et_files,
    "dem": dem_file,
}

In [ ]:
ioh.TimeSeriesInputConfig.model_validate(config)

In [ ]:
et_ts, radiation_ts, et_sd, dem = ioh.read_input(config)

In [ ]:
print(et_ts)
print(radiation_ts)
print(et_sd)
print(dem)

In [ ]:
config = {
    "et_time_series": et_ts_files,
}

In [ ]:
ioh.TimeSeriesInputConfig.model_validate(config)

In [ ]:
et_ts, radiation_ts, et_sd, dem = ioh.read_input(config)

In [ ]:
print(et_ts)
print(radiation_ts)
print(et_sd)
print(dem)

### Configure timeseries over a period

In [ ]:
config = {
    "period_start": "2025-08-23",
    "period_end": "2025-08-29",
    "window": 3,
    "shift": 2,
    "et_time_series_dir": output_dir,
    "radiation_dir": output_dir,
    "et_single_date_dir": output_dir,
}

In [ ]:
ioh.WindowTimeSeriesInputConfig.model_validate(config)